##### Import libraries

In [1]:
import IPython
import pandas
import sparqldataframe
from SPARQLWrapper import SPARQLWrapper, JSON, CSV
import os 
import subprocess
import time
import pandas as pd 

##### Useful functions

In [2]:
def displaySparqlResults(results):
    '''
    Displays as HTML the result of a SPARQLWrapper query in a Jupyter notebook.
    
        Parameters:
            results (dictionnary): the result of a call to SPARQLWrapper.query().convert()
    '''
    variableNames = results['head']['vars']
    tableCode = '<table><tr><th>{}</th></tr><tr>{}</tr></table>'.format('</th><th>'.join(variableNames), '</tr><tr>'.join('<td>{}</td>'.format('</td><td>'.join([row[vName]['value'] if vName in row.keys() else "&nbsp;" for vName in variableNames]))for row in results["results"]["bindings"]))
    IPython.display.display(IPython.display.HTML(tableCode))

##### Define file paths and prefixes

In [5]:
reactomeVersion = 95
prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>

PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/>
PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>

PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>

PREFIX reactome: <http://www.reactome.org/biopax/{}/48887#>
""".format(reactomeVersion)

biopaxURI = "http://www.biopax.org/release/biopax-level3.owl#"
reactomeURI = "http://www.reactome.org/biopax/{}#".format(reactomeVersion)
uniprotURI = "http://purl.uniprot.org/uniprot/"

current_directory = os.getcwd()
endpoint_reactome = "http://localhost:3030/reactome/query"
rdfFormat = "turtle"
BioPAX_Ontology_file_path = os.path.join(current_directory, '../', 'Data', 'biopax_ontology/biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../', 'Data', 'reactome/Homo_sapiens_v95.owl')

##### Launch SPARQL endpoint

In [6]:
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', ReactomeBioPAX_file_path,
    '--file', BioPAX_Ontology_file_path,
    '/reactome']

process = subprocess.Popen(command)
time.sleep(60)

19:01:30 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/../Data/reactome/Homo_sapiens_v95.owl
19:01:32 WARN  riot            :: [line: 71253, col: 45] {W137} Input is large. Switching off checking for illegal reuse of rdf:ID's.
19:01:52 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/../Data/biopax_ontology/biopax-level3.owl
19:01:52 INFO  Server          :: Running in read-only mode for /reactome
19:01:52 INFO  Server          :: Apache Jena Fuseki 4.9.0
19:01:53 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
19:01:53 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/run
19:01:53 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/run/shiro.ini
19:01:53 INFO  Server          :: Database: in-memory, with files loaded
19:0

##### Query the BioPAX export of Reactome to get the list of all ProteinReferences and SmallMoleculeReferences

In [7]:
query="""
SELECT DISTINCT ?entityRef ?entityRefName ?entityID
WHERE {
  VALUES ?entityRefType { bp3:ProteinReference bp3:SmallMoleculeReference }
  ?entityRef rdf:type ?entityRefType .
  ?entityRef bp3:xref ?entityRefXref .
  ?entityRefXref bp3:id ?entityID .
  ?entityRef bp3:name ?entityRefName .
}
"""
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
#displaySparqlResults(results)

sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens95/UtilityFiles/ReactomeHomoSapiens95EntityRefs.csv", "wb") as f:
    f.write(results)

19:02:33 INFO  Fuseki          :: [1] GET http://localhost:3030/reactome/query?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0A%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0A%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0A%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/95/48887%23%3E%0A%0ASELECT+DISTINCT+%3FentityRef+%3FentityRefName+%3FentityID%0AWHERE+%7B%0A++VALUES+%3FentityRefType+%7B+bp3%3AProteinR

##### Generate node table of SIF abstraction of Reactome

In [ ]:
SIFabstraction = pd.read_csv("../Results/ReactomeHomoSapiens95/Meaning2/ReactomeHomoSapiens95.csv", sep=",", header=None)
print(SIFabstraction.head())

NodeTableSIF = pd.DataFrame(columns=['Node', 'Type', 'EntityName'])
SIFentities = list()

for index, row in SIFabstraction.iterrows():
    if not row[0] in SIFentities:
        SIFentities.append(row[0])
    if not row[2] in SIFentities:
        SIFentities.append(row[2])

print(len(SIFentities))
print(SIFentities)

NodeTableSIF['Node'] = SIFentities
NodeTypes = list()
for index, row in NodeTableSIF.iterrows():
    if "Protein" in row[0]:
        NodeTypes.append("Protein")
    elif "SmallMolecule" in row[0]:
        NodeTypes.append("SmallMolecule")
NodeTableSIF['Type'] = NodeTypes

print(NodeTableSIF)

                       0                          1                     2
0   reactome:Protein4607  abstraction:InComplexWith  reactome:Protein4626
1   reactome:Protein7364  abstraction:InComplexWith  reactome:Protein7403
2   reactome:Protein4225  abstraction:InComplexWith  reactome:Protein4373
3   reactome:Protein7334  abstraction:InComplexWith  reactome:Protein7342
4  reactome:Protein30097  abstraction:InComplexWith  reactome:Protein9028


KeyboardInterrupt: 

In [ ]:
PathwayEntities = pd.read_csv("../Results/ReactomeHomoSapiens95/UtilityFiles/ReactomeHomoSapiens95Entity.csv", sep=",", header=0)

dicoEntityNames = dict()
for item,row in PathwayEntities.iterrows():
    entityURI = f"reactome:{row[0][40:]}"
    dicoEntityNames[entityURI] = row[1]

SIFentityNames = list()
for index, row in NodeTableSIF.iterrows():
    entity = row[0]
    SIFentityNames.append(dicoEntityNames[entity])

NodeTableSIF['EntityName'] = SIFentityNames
print(NodeTableSIF)
print(NodeTableSIF.head())

/tmp/ipykernel_631738/3159724472.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  entityURI = f"reactome:{row[0][40:]}"
/tmp/ipykernel_631738/3159724472.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dicoEntityNames[entityURI] = row[1]
/tmp/ipykernel_631738/3159724472.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  entity = row[0]


                             Node           Type     EntityName
0            reactome:Protein4607        Protein          WDR46
1            reactome:Protein4626        Protein       UTP14A,C
2            reactome:Protein7364        Protein         MRPL57
3            reactome:Protein7403        Protein         MRPL42
4            reactome:Protein4225        Protein            p68
...                           ...            ...            ...
17780   reactome:SmallMolecule990  SmallMolecule  A antigen-sec
17781  reactome:SmallMolecule4545  SmallMolecule  Fru, Gal, Glc
17782  reactome:SmallMolecule5263  SmallMolecule  Fru, Gal, Glc
17783  reactome:SmallMolecule5052  SmallMolecule    HOCl, NO2Cl
17784  reactome:SmallMolecule2047  SmallMolecule           RvD5

[17785 rows x 3 columns]
                   Node     Type EntityName
0  reactome:Protein4607  Protein      WDR46
1  reactome:Protein4626  Protein   UTP14A,C
2  reactome:Protein7364  Protein     MRPL57
3  reactome:Protein7403  Prote

In [ ]:
NodeTableSIF.to_csv("../Results/ReactomeHomoSapiens95/UtilityFiles/NodeTableSIF_ReactomeHomoSapiens95_EntityRefs.csv", sep=",", index=False)

In [10]:
process.kill()
time.sleep(60)